# Production Inference Pipeline
Ready to process unlabelled daily data using the `.joblib` weights generated by `training.ipynb`.


In [ ]:
import pandas as pd
import joblib
import os

# 1. Load Serialized Models
try:
    prophet_model = joblib.load('../Models/prophet_model.joblib')
    lgbm_model = joblib.load('../Models/lgbm_model.joblib')
    iso_forest = joblib.load('../Models/iso_forest.joblib')
    print("SUCCESS: Intelligent Weights Loaded.")
except FileNotFoundError:
    print("ERROR: Models not found. You must run training.ipynb first.")


### Execution Engine
Here we load new incoming data, evaluate the baseline, track micro anomalies, and execute the final Hybrid output.


In [ ]:
# 2. Load Incoming Data
# In a true pipeline, this is today's real unlabelled data. We mock it with the processed file.
df_infer = pd.read_csv('../Outputs/dataset_daily_processed.csv')
df_infer['Date'] = pd.to_datetime(df_infer['Date'])
features = ['Day_of_Week', 'Is_Weekend', 'Is_Holiday', 'Avg_Temp', 'Rainfall', 'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_7']

df_clean = df_infer.dropna(subset=features).copy()

# A. Evaluate Mathematical Anomalies 
df_clean['Anomaly_Score'] = iso_forest.predict(df_clean[features])
df_clean['Status'] = df_clean['Anomaly_Score'].apply(lambda x: 'ALERT' if x == -1 else 'OK')

# B. Generate Prophet Baseline
df_p = df_clean[['Date']].rename(columns={'Date': 'ds'})
prophet_preds = prophet_model.predict(df_p)['yhat'].values

# C. Generate LightGBM Micro-Corrections
lgbm_preds = lgbm_model.predict(df_clean[features])

# Final Combination
df_clean['Forecast_MWh'] = prophet_preds + lgbm_preds

print("--- INFERENCE PREVIEW ---")
display(df_clean[['Date', 'Forecast_MWh', 'Status']].tail(5))

df_clean.to_csv('../Outputs/inference_results.csv', index=False)
print("\nTask Completed. Output saved to /Outputs/inference_results.csv")
